# BÀI 6: ĐỐI SOÁT CẤU TRÚC CỘT & GHI LOG CHUYÊN NGHIỆP (BATCH MERGER NÂNG CAO)
## Mục tiêu thực hành:
1. Chuẩn hóa tên cột tự động (xóa khoảng trắng thừa, ánh xạ tên cột biến thể về tên chuẩn).
2. Dùng phép toán tập hợp (`set`) để đối soát cấu trúc cột (**Schema Validation**), phát hiện chính xác file bị thiếu cột bắt buộc.
3. Cấu hình module `logging` chuẩn của Python để ghi nhật ký vận hành đồng thời ra màn hình Console và lưu vào file `batch_process.log`.
4. Phân cấp log theo mức độ (`INFO`, `WARNING`, `ERROR`) phục vụ kiểm toán và xử lý sự cố.

### Ô 1: Giả lập dữ liệu thực tế đa dạng tình huống từ các đơn vị
> Tạo thư mục `du_lieu_nang_cao/` gồm 4 file Excel:
> - `PGD_01_Chuan.xlsx`: Đúng chuẩn 100%.
> - `PGD_02_TenCotKhac.xlsx`: Cán bộ tự đổi tên cột (`Mã tài sản` thay vì `Ma_TS`, có khoảng trắng thừa ` Ten_TS `).
> - `PGD_03_ThieuCot.xlsx`: Thiếu mất cột bắt buộc `Gia_tri`.
> - `PGD_04_FileRong.xlsx`: File có tiêu đề cột nhưng không có dòng dữ liệu nào.

In [ ]:
from pathlib import Path
import pandas as pd

adv_dir = Path("du_lieu_nang_cao")
adv_dir.mkdir(parents=True, exist_ok=True)

# 1. File đúng chuẩn
pd.DataFrame([
    {"Ma_TS": "TS-101", "Ten_TS": "Máy chủ Dell", "So_luong": 1, "Gia_tri": 85000000},
    {"Ma_TS": "TS-102", "Ten_TS": "Switch Cisco", "So_luong": 2, "Gia_tri": 24000000}
]).to_excel(adv_dir / "PGD_01_Chuan.xlsx", index=False)

# 2. File biến thể tên cột & có khoảng trắng thừa
pd.DataFrame([
    {"Mã tài sản": "TS-201", " Ten_TS ": "Laptop HP ProBook", "Số lượng": 3, "Gia_tri": 54000000, "Ghi_chu": "Mới cấp"}
]).to_excel(adv_dir / "PGD_02_TenCotKhac.xlsx", index=False)

# 3. File thiếu cột bắt buộc 'Gia_tri'
pd.DataFrame([
    {"Ma_TS": "TS-301", "Ten_TS": "Máy chiếu Sony", "So_luong": 1}
]).to_excel(adv_dir / "PGD_03_ThieuCot.xlsx", index=False)

# 4. File rỗng (chỉ có header, 0 dòng dữ liệu)
pd.DataFrame(columns=["Ma_TS", "Ten_TS", "So_luong", "Gia_tri"]).to_excel(adv_dir / "PGD_04_FileRong.xlsx", index=False)

print(f"Đã tạo 4 file giả lập tình huống thực tế tại: {adv_dir.resolve()}")

### Ô 2: Thiết lập hệ thống ghi nhật ký (`logging`) chuẩn mực
> Ghi đồng thời ra màn hình Console và lưu vĩnh viễn vào tệp `batch_process.log` với định dạng thời gian chuẩn.

In [ ]:
import logging

# Reset handlers cũ trong Jupyter Notebook để tránh in lặp dòng khi chạy lại ô
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.FileHandler("batch_process.log", mode="w", encoding="utf-8"),
        logging.StreamHandler()
    ]
)

logging.info("Khởi tạo hệ thống ghi log tự động thành công!")

### Ô 3: Xây dựng hàm chuẩn hóa tên cột & đối soát Schema Validation
> Sử dụng từ điển ánh xạ (`COLUMN_MAPPING`) để tự động sửa các lỗi đặt tên cột thường gặp, sau đó dùng phép trừ tập hợp (`REQUIRED_COLS - set(df.columns)`) để kiểm tra tính toàn vẹn.

In [ ]:
# Tập hợp các cột bắt buộc phải có
REQUIRED_COLS = {"Ma_TS", "Ten_TS", "So_luong", "Gia_tri"}

# Từ điển ánh xạ các biến thể tên cột về tên chuẩn
COLUMN_MAPPING = {
    "Mã tài sản": "Ma_TS",
    "Ma tai san": "Ma_TS",
    "Tên tài sản": "Ten_TS",
    "Số lượng": "So_luong",
    "Giá trị": "Gia_tri",
    "Nguyên giá": "Gia_tri"
}

def chuan_hoa_va_kiem_tra(df, file_name):
    # 1. Xóa khoảng trắng thừa ở đầu/cuối tên mọi cột
    df.columns = [str(col).strip() for col in df.columns]
    
    # 2. Đổi tên các cột biến thể về tên chuẩn
    df = df.rename(columns=COLUMN_MAPPING)
    
    # 3. Kiểm tra file rỗng
    if df.empty:
        logging.warning(f"[{file_name}] File không có dòng dữ liệu nào -> Bỏ qua.")
        return None
        
    # 4. Đối soát cột bắt buộc bằng phép trừ tập hợp (Set difference)
    missing_cols = REQUIRED_COLS - set(df.columns)
    if missing_cols:
        logging.error(f"[{file_name}] Thiếu cột bắt buộc: {missing_cols} -> Từ chối nạp!")
        return None
        
    # 5. Chỉ giữ lại đúng các cột chuẩn (loại bỏ cột thừa như 'Ghi_chu' nếu không cần)
    df_clean = df[list(REQUIRED_COLS)].copy()
    return df_clean

### Ô 4: Chạy quy trình hợp nhất tự động và kiểm tra kết quả + file log
> Kết hợp toàn bộ quy trình quét file, chuẩn hóa, đối soát và ghi nhật ký.

In [ ]:
logging.info("=== BẮT ĐẦU QUÁ TRÌNH QUÉT VÀ HỢP NHẤT DỮ LIỆU ===")

valid_dfs = []
files = [f for f in adv_dir.glob("*.xlsx") if not f.name.startswith("~$")]

for f in sorted(files):
    try:
        raw_df = pd.read_excel(f)
        clean_df = chuan_hoa_va_kiem_tra(raw_df, f.name)
        
        if clean_df is not None:
            clean_df["Nguon_Don_Vi"] = f.stem
            valid_dfs.append(clean_df)
            logging.info(f"[{f.name}] Nạp thành công {len(clean_df)} dòng hợp lệ.")
            
    except Exception as e:
        logging.error(f"[{f.name}] Lỗi đọc file nghiêm trọng: {e}")

if valid_dfs:
    df_final = pd.concat(valid_dfs, ignore_index=True)
    # Sắp xếp lại thứ tự cột cho đẹp
    ordered_cols = ["Nguon_Don_Vi", "Ma_TS", "Ten_TS", "So_luong", "Gia_tri"]
    df_final = df_final[ordered_cols]
    logging.info(f"=== HOÀN TẤT: Đã gộp {len(df_final)} dòng từ {len(valid_dfs)}/{len(files)} file ===")
    display(df_final)
else:
    logging.warning("Không có dữ liệu nào đạt chuẩn để tổng hợp!")